<a href="https://colab.research.google.com/github/Rakshitha004/DSP/blob/main/2ndprogram.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit pyngrok pandas bcrypt


In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("31n8F03BarWuSp4oPT50xsH8lqD_2dQWSDsKiRMhiaxmgeKE2")


In [ ]:
%%writefile app.py
import streamlit as st
import hashlib, bcrypt, itertools, math, time
import pandas as pd

# ---------- Utilities ----------
def hash_str(s: str, algo: str) -> str:
    b = s.encode("utf-8")
    if algo == "MD5": return hashlib.md5(b).hexdigest()
    if algo == "SHA1": return hashlib.sha1(b).hexdigest()
    if algo == "SHA256": return hashlib.sha256(b).hexdigest()
    raise ValueError("Unsupported algo")

def verify_hash(candidate: str, target: str, algo: str) -> bool:
    if algo == "Bcrypt ($2b$)":
        try: return bcrypt.checkpw(candidate.encode(), target.encode())
        except: return False
    return hash_str(candidate, algo) == target

def password_strength(pw: str):
    lowers, uppers = any(c.islower() for c in pw), any(c.isupper() for c in pw)
    digits, symbols = any(c.isdigit() for c in pw), any(not c.isalnum() for c in pw)
    variety = sum([lowers, uppers, digits, symbols])
    charset_size = (26 if lowers else 0)+(26 if uppers else 0)+(10 if digits else 0)+(33 if symbols else 0)
    entropy = 0 if charset_size == 0 else round(len(pw)*math.log2(charset_size),1)
    if len(pw)<8 or variety<=1: return "Weak", entropy
    if len(pw)>=12 and variety>=3: return "Strong", entropy
    return "Medium", entropy

def brute_force(charset, min_len, max_len, target, algo, max_attempts=200000):
    total = sum(len(charset)**L for L in range(min_len,max_len+1))
    tried = 0
    for L in range(min_len,max_len+1):
        for tup in itertools.product(charset, repeat=L):
            cand = ''.join(tup)
            tried+=1
            if verify_hash(cand, target, algo):
                return cand, tried, total
            if tried >= max_attempts:  # safety stop
                return None, tried, total
    return None, tried, total

# ---------- Streamlit GUI ----------
st.set_page_config(page_title="Password Attack Lab", page_icon="🔐", layout="wide")
st.title("🔐 Password Attack Lab (Colab GUI)")

mode = st.radio("Target type", ["HASH (MD5/SHA1/SHA256/Bcrypt)", "PLAINTEXT (auto-hash)"], horizontal=True)

algo = "SHA256"
target_hash, target_plain = "", ""

if mode.startswith("HASH"):
    algo = st.selectbox("Hash Algorithm", ["MD5","SHA1","SHA256","Bcrypt ($2b$)"])
    target_hash = st.text_input("Enter Target Hash")
else:
    algo = st.selectbox("Hash Algorithm", ["MD5","SHA1","SHA256"])
    target_plain = st.text_input("Enter any word (auto-hash)")
    if target_plain:
        target_hash = hash_str(target_plain, algo)   # auto-generate hash
        st.write(f"Auto-generated {algo} hash for '{target_plain}': {target_hash}")

dictionary = st.text_area("Dictionary words (one per line)",
                          value="password\n123456\nP@ssw0rd\nadmin\nwelcome\nRakshitha\nR@k$hith@\nidontknow").splitlines()

enable_brute = st.checkbox("Enable brute-force after dictionary attack")
charset = st.text_input("Charset for brute-force", "abc123")
min_len = st.number_input("Min length",1,8,4)
max_len = st.number_input("Max length",min_len,8,5)

if st.button("🚀 Run Attack"):
    results=[]
    found=None
    start=time.time()

    # ----------------- Dictionary Attack with Progress Bar -----------------
    if dictionary:
        progress_bar = st.progress(0)
        total_words = len(dictionary)

        for idx, word in enumerate(dictionary):
            ok = verify_hash(word, target_hash, algo)
            cat, ent = password_strength(word)
            results.append({"candidate": word, "match": ok, "category": cat, "entropy": ent})

            if ok and not found:
                found = word

            # Update progress
            progress_bar.progress((idx + 1) / total_words)

        if found:
            st.success(f"✅ Found in dictionary: {found}")
        else:
            st.warning("❌ Not found in dictionary")

    # ----------------- Brute-force Attack -----------------
    if enable_brute and not found:
        cand, tried, total = brute_force(charset, min_len, max_len, target_hash, algo)
        if cand:
            found=cand
            st.success(f"✅ Found by brute-force: {cand} ({tried}/{total} tried)")
        else:
            st.error(f"❌ Not found in brute-force space ({tried}/{total} tried)")

    df=pd.DataFrame(results)
    if not df.empty:
        st.subheader("📊 Results")
        st.dataframe(df)
        st.download_button("⬇️ Download CSV", data=df.to_csv(index=False).encode(),
                           file_name="analysis.csv", mime="text/csv")
    st.info(f"⏱️ Completed in {time.time()-start:.2f}s")



Overwriting app.py


In [ ]:
#!/usr/bin/env python3
# tiny_pw_attack.py — minimal dictionary + tiny brute attacker
import sys,hashlib,itertools,time

def h(s,a): return hashlib.new(a,s.encode()).hexdigest()

# Check if running in Colab and adjust argument parsing
if '__file__' not in locals():
    # Running in Colab, use hardcoded or default arguments
    algo = "sha256"  # Default algorithm
    target = "a94a8fe5ccb19ba61c4c0873d391e987982fbbd3" # Default target (for "test")
    # You can add more logic here to get arguments from user input if needed
    words = ["password","123456","admin","welcome"] # Default dictionary
    brute_force_enabled = False
    charset = ""
    L = 0
else:
    # Running as a script, use sys.argv
    if len(sys.argv) < 3:
        print("Usage: tiny_pw_attack.py <md5|sha1|sha256> <target_hash> [dictfile] [--brute charset maxlen]"); sys.exit()
    algo,target = (sys.argv[1].lower(), sys.argv[2].lower())
    words = open(sys.argv[3]).read().split() if len(sys.argv)>3 and not sys.argv[3].startswith("--") else ["password","123456","admin","welcome"]
    brute_force_enabled = "--brute" in sys.argv
    if brute_force_enabled:
        i=sys.argv.index("--brute")
        if len(sys.argv) > i+2:
            charset=sys.argv[i+1]
            L=int(sys.argv[i+2])
        else:
            print("Error: --brute requires charset and maxlen arguments"); sys.exit()
    else:
        charset = ""
        L = 0

t0=time.time()
for w in words:
    if h(w,algo)==target: print("FOUND (dict):",w,"time",round(time.time()-t0,2)); sys.exit()

if brute_force_enabled:
    for Llen in range(1,L+1):
        for tup in itertools.product(charset, repeat=Llen):
            c=''.join(tup)
            if h(c,algo)==target: print("FOUND (brute):",c,"time",round(time.time()-t0,2)); sys.exit()

print("NOT FOUND time",round(time.time()-t0,2))

NOT FOUND time 0.0


In [ ]:
#!/usr/bin/env python3
# tiny_pw_attack_fixed.py — minimal + robust
import sys,hashlib,itertools,time

def h(s,algo):
    if algo=="md5": return hashlib.md5(s.encode()).hexdigest()
    if algo=="sha1": return hashlib.sha1(s.encode()).hexdigest()
    if algo=="sha256": return hashlib.sha256(s.encode()).hexdigest()
    raise SystemExit("algo must be: md5 | sha1 | sha256")

if len(sys.argv) < 3:
    print("Usage: tiny_pw_attack_fixed.py <md5|sha1|sha256> <target_hash> [dictfile] [--brute charset maxlen]"); sys.exit()
algo = sys.argv[1].lower()
target = sys.argv[2].lower()

# optional dict file (3rd arg if present and not '--brute')
words = ["password","123456","admin","welcome"]
if len(sys.argv) >= 4 and not sys.argv[3].startswith("--"):
    try:
        words = open(sys.argv[3],"r",encoding="utf-8").read().split()
    except Exception as e:
        print("Could not open dict:", e); sys.exit()

t0=time.time()
for w in words:
    if h(w,algo) == target:
        print("FOUND (dict):",w,"time",round(time.time()-t0,2)); sys.exit()
print("[-] Not in dictionary")

# brute?
if "--brute" in sys.argv:
    i = sys.argv.index("--brute")
    try:
        charset = sys.argv[i+1]; maxlen = int(sys.argv[i+2])
    except:
        print("Usage for brute: --brute <charset> <maxlen>"); sys.exit()
    attempts=0
    for L in range(1, maxlen+1):
        for tup in itertools.product(charset, repeat=L):
            attempts+=1
            cand=''.join(tup)
            if h(cand,algo) == target:
                print("FOUND (brute):",cand,"attempts",attempts,"time",round(time.time()-t0,2)); sys.exit()
    print("NOT FOUND in brute space (attempts",attempts,") time",round(time.time()-t0,2))
else:
    print("[i] Brute disabled. Use --brute <charset> <maxlen> to enable")


SystemExit: algo must be: md5 | sha1 | sha256

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# tiny_pw_attack_interactive.py — interactive (works in notebooks & terminals)
import sys,hashlib,itertools,time

def h(s,algo):
    if algo=="md5": return hashlib.md5(s.encode()).hexdigest()
    if algo=="sha1": return hashlib.sha1(s.encode()).hexdigest()
    if algo=="sha256": return hashlib.sha256(s.encode()).hexdigest()
    raise ValueError("algo must be md5|sha1|sha256")

# If run with args, use them; otherwise prompt interactively
if len(sys.argv) >= 3:
    algo=sys.argv[1].lower(); target=sys.argv[2].lower()
    dictfile = sys.argv[3] if len(sys.argv)>=4 and not sys.argv[3].startswith("--") else None
    brute_args = sys.argv[sys.argv.index("--brute")+1:sys.argv.index("--brute")+3] if "--brute" in sys.argv else None
else:
    algo = input("Algo (md5|sha1|sha256) [md5]: ").strip().lower() or "md5"
    target = input("Target hash (hex) or plaintext (you can enter raw text): ").strip()
    if len(target) < 64 and all(c.isalnum() for c in target):  # heuristic: short -> treat as plaintext
        if input("Treat this as PLAINTEXT and auto-hash? (y/N): ").lower().startswith("y"):
            target = h(target, algo)
            print("[info] auto-hash ->", target)
    dictfile = input("Dict file path (enter to use builtin): ").strip() or None
    brute_on = input("Enable brute? (y/N): ").lower().startswith("y")
    if brute_on:
        charset = input("Brute charset (default abc123): ").strip() or "abc123"
        maxlen = int(input("Max length (default 3): ").strip() or "3")
        brute_args = (charset, maxlen)
    else:
        brute_args = None

# load dictionary
words = ["password","123456","admin","welcome"]
if dictfile:
    try:
        words = open(dictfile,"r",encoding="utf-8").read().split()
    except Exception as e:
        print("Could not open dict:", e); sys.exit()

t0=time.time()
for w in words:
    if h(w,algo) == target:
        print("FOUND (dict):",w,"time",round(time.time()-t0,2)); sys.exit()
print("[-] Not in dictionary")

if brute_args:
    charset, maxlen = brute_args
    attempts=0
    for L in range(1, maxlen+1):
        for tup in itertools.product(charset, repeat=L):
            attempts+=1
            cand=''.join(tup)
            if h(cand,algo) == target:
                print("FOUND (brute):",cand,"attempts",attempts,"time",round(time.time()-t0,2)); sys.exit()
    print("NOT FOUND in brute space (attempts",attempts,") time",round(time.time()-t0,2))
else:
    print("[i] Brute disabled. Done.")


ValueError: algo must be md5|sha1|sha256

In [ ]:
# paste & run this single cell in your notebook
import hashlib, itertools, time

def h(s,algo):
    algo = algo.lower()
    if algo=="md5": return hashlib.md5(s.encode()).hexdigest()
    if algo=="sha1": return hashlib.sha1(s.encode()).hexdigest()
    if algo=="sha256": return hashlib.sha256(s.encode()).hexdigest()
    raise ValueError

# --- interactive prompts (works inside notebook) ---
algo = input("Algo (md5|sha1|sha256) [md5]: ").strip().lower() or "md5"
if algo not in ("md5","sha1","sha256"):
    print("Invalid algo — using md5"); algo="md5"

raw = input("Enter target HASH (hex) or plain text: ").strip()
# heuristics: if looks like a hex of correct length -> treat as hash, else ask to auto-hash
is_hash = all(c in "0123456789abcdefABCDEF" for c in raw) and len(raw) in (32,40,64)
if is_hash:
    target = raw.lower()
else:
    if input("Treat input as PLAINTEXT and auto-hash? (y/N): ").lower().startswith("y"):
        target = h(raw, algo)
        print(f"[info] auto-hash ({algo}): {target}")
    else:
        print("Aborted (no valid target)."); raise SystemExit

# small builtin dictionary
words = ["password","123456","admin","welcome","letmein","P@ssw0rd","rakshitha"]
t0=time.time()
found=None
for w in words:
    if h(w, algo) == target:
        print("FOUND in dict:", w, "time", round(time.time()-t0,2))
        found=True
        break

if not found:
    print("[-] Not in dictionary. You can enable tiny brute-force.")
    if input("Enable brute-force? (y/N): ").lower().startswith("y"):
        charset = input("Charset (default: abc123): ").strip() or "abc123"
        maxlen = int(input("Max length (default: 3): ").strip() or "3")
        attempts=0
        for L in range(1, maxlen+1):
            for tup in itertools.product(charset, repeat=L):
                attempts += 1
                cand = ''.join(tup)
                if h(cand, algo) == target:
                    print("FOUND by brute:", cand, "attempts", attempts, "time", round(time.time()-t0,2))
                    raise SystemExit
        print("NOT FOUND in brute space (attempts", attempts, ") time", round(time.time()-t0,2))
    else:
        print("Done.")


Algo (md5|sha1|sha256) [md5]: md5
Enter target HASH (hex) or plain text: abc
Treat input as PLAINTEXT and auto-hash? (y/N): y
[info] auto-hash (md5): 900150983cd24fb0d6963f7d28e17f72
[-] Not in dictionary. You can enable tiny brute-force.
Enable brute-force? (y/N): y
Charset (default: abc123): abc123
Max length (default: 3): 4
FOUND by brute: abc attempts 51 time 9.8


SystemExit: 